# Photodetectors: ideal, diode, and carrier dynamics

Three notebook-local Circulax components share `optical_in`, `anode`, and `cathode` ports:

| Model | Optical conversion | Electrical characteristics | Internal dynamics |
|---|---|---|---|
| `IdealPhotodetector` | Instantaneous responsivity | Ideal photocurrent source | None |
| `Photodiode` | Instantaneous responsivity | Dark I–V and junction C–V | Junction charge |
| `CarrierPhotodiode` | Carrier generation and collection | Same dark I–V and C–V | Electron/hole populations, screening, recombination |

The parameters below are illustrative, not a fit to a particular device. Models stay in this notebook for review before promotion to `circulax.components.photonic`.

Optical terminals use the existing envelope reference impedance of 1 Ω. They have a wavelength-dependent complex reflection coefficient, so terminal field, incident field, and absorbed power are distinct. Each matched source launches the specified incident power toward one detector; a reflected wave is absorbed by that source. Conventional electrical current is positive **into the anode**; illumination contributes negative anode current. All three models are non-holomorphic.

In [ ]:
import diffrax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax import compile_circuit
from circulax.components.base_component import PhysicsReturn, Signals, component, source
from circulax.components.electronic import Resistor, VoltageSource

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True})
QE = 1.602176634e-19  # C
PORTS = ("optical_in", "anode", "cathode")
RESPONSIVITY = 0.8  # A/W
REFERENCE_WAVELENGTH_NM = 1550.0
REFLECTIVITY = 0.04  # reflected fraction of incident power at the reference wavelength
EXTERNAL_RESPONSIVITY = RESPONSIVITY * (1 - REFLECTIVITY)


## Wavelength-dependent responsivity and reflection

All three models accept `wavelength_nm`. This is the optical carrier wavelength of a narrowband envelope; the transient time coordinate describes modulation, rather than oscillations at the optical carrier frequency.

For this example, define an absorbed-power responsivity profile

$$
\mathcal R_\mathrm{abs}(\lambda)=\mathcal R_0\frac{\lambda}{\lambda_0}
\exp[-\tfrac12((\lambda-\lambda_0)/\sigma_R)^2].
$$

The wavelength factor accounts for photon energy at a fixed conversion efficiency; the Gaussian approximates spectral efficiency. `responsivity` is the value at `center_wavelength_nm`, in A/W of **absorbed** optical power, before carrier-collection losses. These illustrative analytic profiles can later be replaced by measured curves. For unity conversion efficiency, absorbed-power responsivity is $q_e\lambda/(hc)$; the default profile stays below this bound.

Optical reflection is an amplitude coefficient, not a current added arbitrarily to the port:

$$
r(\lambda)=\sqrt{\rho_0}\exp[-\tfrac14((\lambda-\lambda_0)/\sigma_r)^2]
\exp[j(\phi_0+k_\phi(\lambda-\lambda_0))],\qquad
Y_\mathrm{opt}(\lambda)=\frac{1-r}{1+r}\quad (Z_0=1).
$$

`reflectivity` $=\rho_0$ is **power reflectance at the reference wavelength**; `reflection_phase_rad` and `reflection_phase_slope_rad_nm` control field phase. With incident field $a$, reflected field $b=ra$, and terminal field $E=a+b$:

$$
a=E/(1+r),\quad P_\mathrm{inc}=|a|^2,\quad
P_\mathrm{refl}=|r|^2P_\mathrm{inc},\quad
P_\mathrm{abs}=(1-|r|^2)P_\mathrm{inc},\quad
I_\mathrm{ph}=\mathcal R_\mathrm{abs}(\lambda)P_\mathrm{abs}.
$$

This one-port model absorbs all non-reflected power and has no optical transmission port. Valid parameter ranges are positive wavelengths and bandwidths, `responsivity >= 0`, and `0 <= reflectivity < 1`. The defaults give 4% reflectance and an external low-power responsivity of $0.8(1-0.04)=0.768$ A/W at 1550 nm. If starting with measured **external** responsivity per incident watt, first divide it by $1-|r|^2$ to obtain the absorbed-power profile; otherwise reflection would be counted twice.

## 1. Ideal detector

Instantaneous optical-to-electrical conversion uses the shared spectral response and optical admittance. Equal and opposite electrical stamps conserve terminal current. Setting `reflectivity=0` recovers the matched-detector limit.

In [ ]:
def spectral_response(responsivity=0.8, wavelength_nm=1550.0, center_wavelength_nm=1550.0,
                      responsivity_bandwidth_nm=250.0, reflectivity=0.04,
                      reflection_bandwidth_nm=180.0, reflection_phase_rad=0.0,
                      reflection_phase_slope_rad_nm=0.01):
    detuning_nm = wavelength_nm - center_wavelength_nm
    r_abs = responsivity * (wavelength_nm / center_wavelength_nm)
    r_abs *= jnp.exp(-0.5 * (detuning_nm / responsivity_bandwidth_nm)**2)
    reflection = jnp.sqrt(reflectivity) * jnp.exp(-0.25 * (detuning_nm / reflection_bandwidth_nm)**2)
    reflection = reflection * jnp.exp(1j * (reflection_phase_rad + reflection_phase_slope_rad_nm * detuning_nm))
    return r_abs, reflection


def optical_conversion(field, responsivity=0.8, wavelength_nm=1550.0, center_wavelength_nm=1550.0,
                       responsivity_bandwidth_nm=250.0, reflectivity=0.04,
                       reflection_bandwidth_nm=180.0, reflection_phase_rad=0.0,
                       reflection_phase_slope_rad_nm=0.01):
    r_abs, reflection = spectral_response(
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    optical_current = (1 - reflection) / (1 + reflection) * field
    incident = field / (1 + reflection)
    absorbed_power = (1 - jnp.abs(reflection)**2) * jnp.abs(incident)**2
    return optical_current, r_abs * absorbed_power


@component(ports=PORTS, holomorphic=False)
def IdealPhotodetector(
    signals: Signals, responsivity: float = 0.8,
    wavelength_nm: float = 1550.0, center_wavelength_nm: float = 1550.0,
    responsivity_bandwidth_nm: float = 250.0, reflectivity: float = 0.04,
    reflection_bandwidth_nm: float = 180.0, reflection_phase_rad: float = 0.0,
    reflection_phase_slope_rad_nm: float = 0.01,
) -> PhysicsReturn:
    optical_current, i_photo = optical_conversion(signals.optical_in,
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    return {"optical_in": optical_current,
            "anode": -i_photo, "cathode": i_photo}, {}

ideal = IdealPhotodetector(reflectivity=0.0)  # matched limit for these unit checks
f, _ = ideal(optical_in=jnp.sqrt(1e-3) * jnp.exp(0.7j))
np.testing.assert_allclose(f["anode"], -0.8e-3)
np.testing.assert_allclose(f["anode"] + f["cathode"], 0.0)
# Smooth intensity derivatives, including exactly zero field.
def detected_current(field_xy):
    field = field_xy[0] + 1j * field_xy[1]
    return -ideal(optical_in=field)[0]["anode"]
np.testing.assert_allclose(jax.grad(detected_current)(jnp.array([0.03, 0.04])), [0.048, 0.064])
assert np.all(np.isfinite(jax.grad(detected_current)(jnp.zeros(2))))

## 2. Diode I–V, C–V, and dark current

With $V=V_a-V_c$, use an analytic dark profile:

$$
I_\mathrm{dark}(V)=I_s\operatorname{expm1}(V/(n_dV_T))
 +I_\mathrm{leak}\tanh(V/V_\mathrm{leak})+V/R_\mathrm{sh}.
$$

`i_leak` is **additional** reverse leakage; total reverse dark current also includes `i_s` and shunt leakage. This avoids counting a separately specified total dark current twice. The profile excludes avalanche breakdown and is intended for the plotted operating range.

For depletion capacitance use $C(V)=C_0(1-V/V_\mathrm{bi})^{-1/2}$. Integrate it to charge:

$$
Q(V)=2C_0V_\mathrm{bi}\left(1-\sqrt{1-V/V_\mathrm{bi}}\right).
$$

Above `fc * v_bi`, continue Q quadratically so Q, C, and dC/dV remain continuous and the depletion singularity is avoided. Circulax receives **Q**, not `C(V)*V`, which would give the wrong differential capacitance. A measured-profile extension should likewise integrate C(V) before stamping charge.

In [ ]:
def dark_iv(v, i_s=1e-12, ideality=1.5, v_t=0.02585,
            i_leak=1e-9, v_leak=0.2, r_sh=1e12):
    return i_s * jnp.expm1(v / (ideality * v_t)) + i_leak * jnp.tanh(v / v_leak) + v / r_sh


def junction_charge(v, c0=20e-15, v_bi=0.8, fc=0.5):
    v_join = fc * v_bi
    # A complex V is retained for the electrical imaginary-block equations.
    v_depletion = jnp.where(jnp.real(v) <= v_join, v, v_join)
    q_depletion = 2 * c0 * v_bi * (1 - jnp.sqrt(1 - v_depletion / v_bi))
    dv = jnp.where(jnp.real(v) > v_join, v - v_join, 0.0)
    c_join = c0 / jnp.sqrt(1 - fc)
    dc_join = c0 / (2 * v_bi * (1 - fc)**1.5)
    return q_depletion + c_join * dv + 0.5 * dc_join * dv**2


@component(ports=PORTS, holomorphic=False)
def Photodiode(signals: Signals, responsivity: float = 0.8,
    wavelength_nm: float = 1550.0, center_wavelength_nm: float = 1550.0,
    responsivity_bandwidth_nm: float = 250.0, reflectivity: float = 0.04,
    reflection_bandwidth_nm: float = 180.0, reflection_phase_rad: float = 0.0,
    reflection_phase_slope_rad_nm: float = 0.01,
               i_s: float = 1e-12, ideality: float = 1.5, v_t: float = 0.02585,
               i_leak: float = 1e-9, v_leak: float = 0.2, r_sh: float = 1e12,
               c0: float = 20e-15, v_bi: float = 0.8, fc: float = 0.5) -> PhysicsReturn:
    v = signals.anode - signals.cathode
    current = dark_iv(v, i_s, ideality, v_t, i_leak, v_leak, r_sh)
    optical_current, i_photo = optical_conversion(signals.optical_in,
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    current -= i_photo
    charge = junction_charge(v, c0, v_bi, fc)
    return {"optical_in": optical_current, "anode": current, "cathode": -current}, {
        "anode": charge, "cathode": -charge}


v_iv = jnp.linspace(-5.0, 0.65, 400)
v_cv = jnp.linspace(-5.0, 0.6, 400)
capacitance = jax.vmap(jax.grad(junction_charge))(v_cv)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for power in [0.0, 1e-6, 1e-3]:
    axes[0].plot(v_iv, dark_iv(v_iv) - EXTERNAL_RESPONSIVITY * power, label=f"P = {power:g} W")
axes[0].set(xlabel="Anode − cathode voltage (V)", ylabel="Anode current (A)", yscale="symlog")
axes[0].set_yscale("symlog", linthresh=1e-10)
axes[0].legend()
axes[1].plot(v_cv, capacitance * 1e15)
axes[1].set(xlabel="Anode − cathode voltage (V)", ylabel="Differential capacitance (fF)")
plt.tight_layout()
plt.show()

v_reverse = jnp.linspace(-5.0, 0.0, 30)
np.testing.assert_allclose(jax.vmap(jax.grad(junction_charge))(v_reverse),
                           20e-15 / jnp.sqrt(1 - v_reverse / 0.8), rtol=1e-12)
assert bool(jnp.all(capacitance > 0))
assert float(dark_iv(-3.0)) < 0

## 3. Electron and hole populations

Represent excess concentrations as $n=N_0x_e$ and $p=N_0x_h$, with normalized states `x_e` and `x_h`. The normalization keeps circuit unknowns near unity. For active volume $\mathcal V$, responsivity defines an effective pair-generation rate:

$$
g(P)=\frac{\mathcal R_\mathrm{abs}(\lambda)P_\mathrm{abs}}{q_e\mathcal V N_0},\qquad
\dot x_e=g-x_e/\tau_e-r,\qquad
\dot x_h=g-x_h/\tau_h-r,\qquad
r=x_ex_h/\tau_\mathrm{rec}.
$$

The absorbed-power responsivity here is the conversion **before collection losses**; wavelength-dependent reflection is applied before generation. Recombination removes pairs from both populations at the same rate. To approximate reduced collection field, use

$$
\tau_{e,h}=\tau_{e,h,0}\frac{V_\mathrm{bi}+V_\mathrm{ref}}
 {V_\mathrm{bi}+\max(-V,0)}
 \left[1+s\frac{\max(x_e,0)+\max(x_h,0)}{2x_\mathrm{screen}}\right].
$$

Equivalently, with a fixed illustrative depletion width $L=1\,\mu\mathrm m$, expose the internal field explicitly:

$$
E_0=(V_\mathrm{bi}+\max(-V,0))/L,\quad
E_\mathrm{eff}=E_0/[1+s(x_e+x_h)/(2x_\mathrm{screen})],\quad
\tau_{e,h}=\tau_{e,h,0}E_\mathrm{ref}/E_\mathrm{eff}.
$$

**Feedback loop:** more carriers → smaller internal field → longer sweep-out times → more carriers remaining in the absorber. Optical generation supplies the carriers and recombination limits their accumulation. The field is an algebraic function of carrier states and actual junction voltage, rather than an extra independent dynamic state.

This density-based screening law is phenomenological: a spatially averaged pair of populations cannot reconstruct the actual space-charge field. Different collection rates and field screening are motivated by [self-consistent transport simulations](https://www.sciencedirect.com/science/article/abs/pii/S0038110100000952). Fit its parameters to measured impulse responses before using it for device predictions. The reverse-bias branch and population clamps are piecewise differentiable; the examples stay in reverse bias.

Terminal photocurrent is approximated by a weighted transport sum:

$$
I_\mathrm{ph}=q_e\mathcal V N_0
 [w_e x_e/\tau_e+(1-w_e)x_h/\tau_h].
$$

The example uses `electron_weight = 0.5`; weights sum to one so full collection contributes **one electron charge per pair**, not two. They approximate weighting-field travel fractions, not two independent collected terminal currents. Fixed sweep-out times alone give linear filtering. Density-dependent sweep-out produces peak compression and longer recovery; recombination also reduces integrated charge.

Circulax solves `f + dQ/dt = 0`. State rows therefore stamp `tau_ref * (loss − generation)` in f and `tau_ref * x` in Q. The stored state quantities are scaled rate-equation variables, not additional terminal junction charge.

In [ ]:
def internal_field(x_e, x_h, v, v_bi=0.8, screening=1.0,
                   x_screen=1.0, depletion_width_m=1e-6):
    population = (jnp.maximum(jnp.real(x_e), 0) + jnp.maximum(jnp.real(x_h), 0)) / 2
    reverse_bias = jnp.maximum(-jnp.real(v), 0)
    unscreened = (v_bi + reverse_bias) / depletion_width_m
    effective = unscreened / (1 + screening * population / x_screen)
    return unscreened, effective


def carrier_rates(x_e, x_h, v, tau_e0=8e-12, tau_h0=40e-12,
                  v_bi=0.8, v_ref=3.0, screening=1.0, x_screen=1.0,
                  tau_rec=500e-12):
    _, effective_field = internal_field(x_e, x_h, v, v_bi, screening, x_screen)
    reference_field = (v_bi + v_ref) / 1e-6
    tau_e, tau_h = tau_e0 * reference_field / effective_field, tau_h0 * reference_field / effective_field
    recombination = x_e * x_h / tau_rec
    return tau_e, tau_h, recombination


@component(ports=PORTS, states=("x_e", "x_h"), holomorphic=False)
def CarrierPhotodiode(
    signals: Signals, responsivity: float = 0.8,
    wavelength_nm: float = 1550.0, center_wavelength_nm: float = 1550.0,
    responsivity_bandwidth_nm: float = 250.0, reflectivity: float = 0.04,
    reflection_bandwidth_nm: float = 180.0, reflection_phase_rad: float = 0.0,
    reflection_phase_slope_rad_nm: float = 0.01,
    i_s: float = 1e-12, ideality: float = 1.5, v_t: float = 0.02585,
    i_leak: float = 1e-9, v_leak: float = 0.2, r_sh: float = 1e12,
    c0: float = 20e-15, v_bi: float = 0.8, fc: float = 0.5,
    volume_m3: float = 100e-18, density_scale_m3: float = 1e21,
    tau_e0: float = 8e-12, tau_h0: float = 40e-12, v_ref: float = 3.0,
    screening: float = 1.0, x_screen: float = 1.0, tau_rec: float = 500e-12,
    electron_weight: float = 0.5, tau_ref: float = 20e-12,
) -> PhysicsReturn:
    v = signals.anode - signals.cathode
    x_e, x_h = signals.x_e, signals.x_h
    charge_scale = QE * volume_m3 * density_scale_m3
    optical_current, generation_current = optical_conversion(signals.optical_in,
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    generation = generation_current / charge_scale
    tau_e, tau_h, recombination = carrier_rates(
        x_e, x_h, v, tau_e0, tau_h0, v_bi, v_ref, screening, x_screen, tau_rec)
    i_photo = charge_scale * (electron_weight * x_e / tau_e + (1 - electron_weight) * x_h / tau_h)
    current = dark_iv(v, i_s, ideality, v_t, i_leak, v_leak, r_sh) - i_photo
    charge = junction_charge(v, c0, v_bi, fc)
    return {
        "optical_in": optical_current, "anode": current, "cathode": -current,
        "x_e": tau_ref * (x_e / tau_e + recombination - generation),
        "x_h": tau_ref * (x_h / tau_h + recombination - generation),
    }, {"anode": charge, "cathode": -charge,
        "x_e": tau_ref * x_e, "x_h": tau_ref * x_h}

## A common circuit testbench

Three independent matched optical-envelope sources launch the **same incident power** toward their respective detectors, as replicated experiments rather than a power splitter. Each source absorbs any returning reflection. A source stamps `I = E − 2a` and each detector stamps `I = Y_opt E`; their shared node solves to `E = (1 + r)a`. This preserves the prescribed incident power when reflection changes with wavelength. Each cathode is biased positively and each anode drives a 50 Ω load to ground. The ideal detector has no dark current; the two diode models share identical electrical profiles.

Use a Gaussian optical **power** pulse with 12 ps standard deviation (28.3 ps FWHM) to approximate an impulse. Run through 1.5 ns so even the screened carrier tail can be integrated. Outputs remain in volts; load current is output voltage / 50 Ω.

In [ ]:
@source(ports=("p1", "p2"), holomorphic=True)
def OpticalGaussianPulse(signals: Signals, t: float, power: float = 1e-3,
                         center: float = 120e-12, sigma: float = 12e-12,
                         phase: float = 0.0) -> PhysicsReturn:
    incident = jnp.sqrt(power) * jnp.exp(-0.25 * ((t - center) / sigma)**2 + 1j * phase)
    current = signals.p1 - signals.p2 - 2 * incident
    return {"p1": current, "p2": -current}, {}


models = {"ground": lambda: 0, "optical_pulse": OpticalGaussianPulse,
          "bias": VoltageSource, "load": Resistor,
          "ideal": IdealPhotodetector, "diode": Photodiode, "carrier": CarrierPhotodiode}
instances = {"GND": {"component": "ground"},
             "BIAS": {"component": "bias", "settings": {"V": 3.0}}}
connections = {
    "GND,p1": ("LIGHT_ideal,p2", "LIGHT_diode,p2", "LIGHT_carrier,p2",
               "BIAS,p2", "R_ideal,p2", "R_diode,p2", "R_carrier,p2"),
    "BIAS,p1": ("PD_ideal,cathode", "PD_diode,cathode", "PD_carrier,cathode"),
}
for name in ("ideal", "diode", "carrier"):
    instances[f"LIGHT_{name}"] = {"component": "optical_pulse"}
    connections[f"LIGHT_{name},p1"] = f"PD_{name},optical_in"
    instances[f"PD_{name}"] = {"component": name}
    instances[f"R_{name}"] = {"component": "load", "settings": {"R": 50.0}}
    connections[f"PD_{name},anode"] = f"R_{name},p1"
netlist = {"instances": instances, "connections": connections,
           "ports": {"light": "LIGHT_ideal,p1", **{name: f"PD_{name},anode" for name in ("ideal", "diode", "carrier")}}}
circuit = compile_circuit(netlist, models, is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)
times = jnp.linspace(0, 1.5e-9, 1501)


def simulate(power=1e-3, bias=3.0, screening=1.0, tau_rec=500e-12,
             tau_e0=8e-12, tau_h0=40e-12, dt=1e-12, wavelength_nm=1550.0, reflectivity=0.04):
    updates = {"power": power, "BIAS.V": bias, "wavelength_nm": wavelength_nm,
               "reflectivity": reflectivity,
               "PD_carrier.screening": screening, "PD_carrier.tau_rec": tau_rec,
               "PD_carrier.tau_e0": tau_e0, "PD_carrier.tau_h0": tau_h0}
    y0 = circuit.dc(params=updates)
    solution = circuit.transient(t0=0, t1=float(times[-1]), dt0=dt, y0=y0,
                                 params=updates, saveat=diffrax.SaveAt(ts=times),
                                 max_steps=5000, throw=True)
    currents = {name: np.real((circuit.port(solution.ys, name) - circuit.port(solution.ys, "GND,p1"))
                                - (circuit.port(y0, name) - circuit.port(y0, "GND,p1"))) / 50
                for name in ("ideal", "diode", "carrier")}
    concentrations = {state: np.real(circuit.port(solution.ys, f"PD_carrier,{state}")) * 1e21
                      for state in ("x_e", "x_h")}
    junction_v = circuit.port(solution.ys, "carrier") - circuit.port(solution.ys, "BIAS,p1")
    field0, field = internal_field(concentrations["x_e"] / 1e21, concentrations["x_h"] / 1e21,
                                   junction_v, screening=screening)
    concentrations["field_unscreened"] = np.asarray(field0)
    concentrations["field_screened"] = np.asarray(field)
    assert np.all(np.isfinite(solution.ys))
    assert max(np.max(np.abs(np.imag(circuit.port(solution.ys, name))))
               for name in ("ideal", "diode", "carrier")) < 1e-9
    return currents, concentrations

currents, concentrations = simulate()
t_ps = np.asarray(times) * 1e12
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, current in currents.items():
    axes[0].plot(t_ps, current * 1e3, label=name)
axes[0].set(xlabel="Time (ps)", ylabel="Baseline-subtracted load current (mA)", xlim=(60, 600))
axes[0].legend()
for state in ("x_e", "x_h"):
    density = concentrations[state]
    axes[1].plot(t_ps, density / 1e6, label="electrons" if state == "x_e" else "holes")
axes[1].set(xlabel="Time (ps)", ylabel="Excess concentration (cm⁻³)", xlim=(60, 600))
axes[1].legend()
plt.tight_layout()
plt.show()

## Saturation, bias dependence, and asymmetric response

Compare normalized responses across pulse powers to distinguish linear filtering from power-dependent dynamics. Report 10–90% rising-edge and 90–10% falling-edge times using interpolated first crossings around the global peak. These are response-shape metrics for this finite Gaussian stimulus, not intrinsic device constants.

In [ ]:
def crossing_time(t, y, level):
    hits = np.flatnonzero((y[:-1] - level) * (y[1:] - level) <= 0)
    if not len(hits):
        return np.nan
    k = hits[0]
    if y[k + 1] == y[k]:
        return t[k]
    return t[k] + (level - y[k]) * (t[k + 1] - t[k]) / (y[k + 1] - y[k])


def pulse_metrics(current, power, wavelength_nm=1550.0, reflectivity=0.04):
    t = np.asarray(times)
    k = int(np.argmax(current))
    peak = current[k]
    rise = crossing_time(t[:k + 1], current[:k + 1], 0.9 * peak) - crossing_time(t[:k + 1], current[:k + 1], 0.1 * peak)
    fall = crossing_time(t[k:], current[k:], 0.1 * peak) - crossing_time(t[k:], current[k:], 0.9 * peak)
    r_abs, reflection = spectral_response(wavelength_nm=wavelength_nm, reflectivity=reflectivity)
    external_responsivity = float(r_abs * (1 - jnp.abs(reflection)**2))
    incident_charge = external_responsivity * power * np.sqrt(2 * np.pi) * 12e-12
    return peak, np.trapezoid(current, t) / incident_charge, rise * 1e12, fall * 1e12


powers = np.array([1e-5, 1e-3, 1e-2])
sweep = {1e-3: (currents, concentrations)}
for power in powers:
    if power not in sweep:
        sweep[power] = simulate(power=power)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
print("Peak P (mW)   Peak I (mA)   Charge fraction   Rise (ps)   Fall (ps)")
for power in powers:
    current = sweep[power][0]["carrier"]
    peak, fraction, rise, fall = pulse_metrics(current, power)
    print(f"{power * 1e3:11.3g} {peak * 1e3:13.4g} {fraction:17.4f} {rise:11.2f} {fall:11.2f}")
    axes[0].plot(t_ps, current / (EXTERNAL_RESPONSIVITY * power), label=f"{power * 1e3:g} mW")
    axes[1].plot(t_ps, current / peak, label=f"{power * 1e3:g} mW")
axes[0].set(xlabel="Time (ps)", ylabel="I / (external responsivity × peak power)", xlim=(60, 800))
axes[1].set(xlabel="Time (ps)", ylabel="I / peak I", xlim=(60, 800))
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

low_bias, _ = simulate(power=1e-3, bias=1.0)
plt.figure()
plt.plot(t_ps, currents["carrier"] * 1e3, label="3 V reverse bias")
plt.plot(t_ps, low_bias["carrier"] * 1e3, label="1 V reverse bias")
plt.xlim(60, 800)
plt.xlabel("Time (ps)")
plt.ylabel("Load current (mA)")
plt.legend()
plt.tight_layout()
plt.show()

## Isolating the nonlinear feedback

Keep recombination negligible in both experiments to isolate screening. At high optical power, compare screening on and off with identical bias, carrier times, diode profile, and load. With screening active, the field collapses during carrier buildup, sweep-out slows, and the pulse develops a long tail. Peak compression can occur while almost all charge is eventually collected; this differs from recombination loss.


The optical stimulus is symmetric. Separate electron and hole sweep-out times already cause a fast rise and slower fall at low power. The comparison below isolates the additional asymmetry produced by screening; the final plot marks both threshold intervals explicitly.

In [ ]:
feedback_off, off_states = simulate(power=1e-2, screening=0.0, tau_rec=1e6)
feedback_on, on_states = simulate(power=1e-2, screening=1.0, tau_rec=1e6)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for label, current_set, states in [("screening off", feedback_off, off_states),
                                   ("screening on", feedback_on, on_states)]:
    axes[0].plot(t_ps, current_set["carrier"] * 1e3, label=label)
    axes[1].plot(t_ps, states["field_screened"] / 1e6, label=label)
    axes[2].plot(t_ps, (states["x_e"] + states["x_h"]) / 1e6, label=label)
axes[0].set(ylabel="Load current (mA)")
axes[1].set(ylabel="Effective internal field (MV/m)")
axes[2].set(ylabel="Total excess concentration (cm⁻³)")
for ax in axes:
    ax.set(xlabel="Time (ps)", xlim=(60, 1200))
    ax.legend()
plt.tight_layout()
plt.show()

screening_ratio = on_states["field_screened"] / on_states["field_unscreened"]
assert np.min(screening_ratio) < 0.5
np.testing.assert_allclose(off_states["field_screened"], off_states["field_unscreened"])
assert np.max(feedback_on["carrier"]) < np.max(feedback_off["carrier"])
print(f"Minimum screened / unscreened internal field: {np.min(screening_ratio):.3f}")
for label, current_set in [("off", feedback_off), ("on", feedback_on)]:
    peak, fraction, rise, fall = pulse_metrics(current_set["carrier"], 1e-2)
    print(f"Screening {label}: peak={peak * 1e3:.3f} mA; charge fraction={fraction:.4f}; rise/fall={rise:.1f}/{fall:.1f} ps")

# Show rise/fall intervals directly, using the same symmetric optical stimulus.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
shape_metrics = {}
optical_shape = np.exp(-0.5 * ((np.asarray(times) - 120e-12) / 12e-12)**2)
for ax, label, current_set in zip(axes, ("screening off", "screening on"),
                                 (feedback_off, feedback_on), strict=True):
    current = current_set["carrier"]
    k = int(np.argmax(current))
    normalized = current / current[k]
    rise10 = crossing_time(t_ps[:k + 1], normalized[:k + 1], 0.1)
    rise90 = crossing_time(t_ps[:k + 1], normalized[:k + 1], 0.9)
    fall90 = crossing_time(t_ps[k:], normalized[k:], 0.9)
    fall10 = crossing_time(t_ps[k:], normalized[k:], 0.1)
    rise, fall = rise90 - rise10, fall10 - fall90
    shape_metrics[label] = rise, fall
    ax.plot(t_ps, normalized, color="0.15", lw=2, label="normalized photocurrent")
    ax.plot(t_ps, optical_shape, "--", color="0.6", label="normalized optical power")
    ax.axvspan(rise10, rise90, alpha=0.2, color="C0", label=f"10–90% rise: {rise:.1f} ps")
    ax.axvspan(fall90, fall10, alpha=0.2, color="C1", label=f"90–10% fall: {fall:.1f} ps")
    ax.scatter([rise10, rise90, fall90, fall10], [0.1, 0.9, 0.9, 0.1], color="0.15", s=24, zorder=3)
    ax.set(title=f"{label}: fall / rise = {fall / rise:.1f}", xlabel="Time (ps)",
           ylabel="Normalized power or current", xlim=(60, 650), ylim=(-0.03, 1.08))
    ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

rise_off, fall_off = shape_metrics["screening off"]
rise_on, fall_on = shape_metrics["screening on"]
assert fall_on > 3 * fall_off
assert fall_on / rise_on > fall_off / rise_off
print(f"Fall/rise ratio: {fall_off / rise_off:.2f} without screening, {fall_on / rise_on:.2f} with screening")


## Conservation and numerical checks

First switch off screening and effectively switch off recombination, and verify that unequal sweep-out times redistribute the impulse without doubling its charge. Then compare a half-size time step, check phase independence, and compare the nonlinear power sweep. Baseline subtraction removes the static dark current, so integrated load charge can be compared with external responsivity × incident pulse energy; the junction returns to its initial charge at the integration endpoints.

In [ ]:
linear_currents, linear_densities = simulate(power=1e-5, screening=0.0, tau_rec=1e6)
linear_metrics = pulse_metrics(linear_currents["carrier"], 1e-5)
np.testing.assert_allclose(linear_metrics[1], 1.0, rtol=0.01)
np.testing.assert_allclose(pulse_metrics(currents["ideal"], 1e-3)[1], 1.0, rtol=0.002)
np.testing.assert_allclose(pulse_metrics(currents["diode"], 1e-3)[1], 1.0, rtol=0.005)

refined, _ = simulate(dt=0.5e-12)
relative_error = np.max(np.abs(refined["carrier"] - currents["carrier"])) / np.max(refined["carrier"])
assert relative_error < 0.01, relative_error

low_peak, low_charge, *_ = pulse_metrics(sweep[1e-5][0]["carrier"], 1e-5)
high_peak, high_charge, high_rise, high_fall = pulse_metrics(sweep[1e-2][0]["carrier"], 1e-2)
assert high_peak / 1e-2 < low_peak / 1e-5  # peak-current compression
assert 0 < high_charge < low_charge <= 1.01  # recombination reduces collected charge
assert high_fall > high_rise
assert np.max(sweep[1e-2][1]["x_h"][-1:]) < 1e17  # return near dark population (m^-3)

# Component-level invariants with populated states and arbitrary optical phase.
pd = CarrierPhotodiode()
f0, q0 = pd(optical_in=np.sqrt(1e-3), anode=0.0, cathode=3.0, x_e=0.5, x_h=1.0)
f1, q1 = pd(optical_in=np.sqrt(1e-3) * np.exp(1.2j), anode=0.0, cathode=3.0, x_e=0.5, x_h=1.0)
for key in f0:
    np.testing.assert_allclose(f0[key], f1[key], rtol=1e-12) if key != "optical_in" else None
np.testing.assert_allclose(f0["anode"] + f0["cathode"], 0)
np.testing.assert_allclose(q0["anode"] + q0["cathode"], 0)
assert min(np.min(density) for _, densities in sweep.values() for state, density in densities.items() if state in ("x_e", "x_h")) > -1e16
print(f"Linear collected charge: {linear_metrics[1]:.6f} × external responsivity × incident pulse energy")
print(f"Half-step peak-relative waveform error: {relative_error:.3%}")
print(f"High-power rise/fall: {high_rise:.2f} / {high_fall:.2f} ps")
print("All notebook checks passed.")

## Spectral response and reflected waves

Sweep the common `wavelength_nm` parameter through all three detector models at low optical power. Place the Gaussian source center at time zero for an illuminated DC operating point. The three independent sources preserve the same incident power at every wavelength.

Plot absorbed-power versus external responsivity, power reflectance, and reflected-field phase. The circuit's solved optical field must satisfy $E=(1+r)a$; extracting $b=E-a$ verifies both reflection magnitude and phase. Separately verify $P_\mathrm{inc}=P_\mathrm{refl}+P_\mathrm{abs}$ and that the ideal photocurrent agrees with the spectral profile. Electrical dark-current subtraction and recombination explain the diode models' slight low-power differences.

In [ ]:
wavelengths_nm = jnp.linspace(1200, 1800, 121)
spectral_power = 1e-6
r_abs, reflection = spectral_response(wavelength_nm=wavelengths_nm)
external_r = r_abs * (1 - jnp.abs(reflection)**2)
dark_op = circuit.dc(params={"power": 0.0}, rtol=1e-10, atol=1e-13)
spectral_op = circuit.dc(params={"power": spectral_power, "center": 0.0,
                                 "wavelength_nm": wavelengths_nm}, rtol=1e-10, atol=1e-13)
measured_r = {
    name: np.real((circuit.port(spectral_op, name) - circuit.port(spectral_op, "GND,p1"))
                  - (circuit.port(dark_op, name) - circuit.port(dark_op, "GND,p1"))) / (50 * spectral_power)
    for name in ("ideal", "diode", "carrier")
}
incident = np.sqrt(spectral_power)
field = circuit.port(spectral_op, "light")
reflected = field - incident
np.testing.assert_allclose(reflected / incident, reflection, rtol=1e-7, atol=1e-9)
np.testing.assert_allclose(measured_r["ideal"], external_r, rtol=1e-7, atol=1e-9)

optical_current, conversion_current = optical_conversion(field, wavelength_nm=wavelengths_nm)
absorbed = jnp.real(field * jnp.conj(optical_current))
np.testing.assert_allclose(absorbed + jnp.abs(reflected)**2, spectral_power, rtol=1e-7)
np.testing.assert_allclose(conversion_current, r_abs * absorbed, rtol=1e-7)
assert bool(jnp.all(absorbed >= 0))
# Physical upper bound for the illustrative absorbed-power conversion efficiency.
quantum_limit = QE * wavelengths_nm * 1e-9 / (6.62607015e-34 * 299792458.0)
assert bool(jnp.all((r_abs >= 0) & (r_abs <= quantum_limit)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(wavelengths_nm, r_abs, "--", color="0.4", label="absorbed-power profile")
for name, response in measured_r.items():
    axes[0].plot(wavelengths_nm, response, label=f"external: {name}",
                 linestyle={"ideal": "-", "diode": "--", "carrier": ":"}[name])
axes[0].set(ylabel="Responsivity (A/W)")
axes[1].plot(wavelengths_nm, jnp.abs(reflection)**2 * 100, label="profile")
axes[1].plot(wavelengths_nm[::8], np.asarray(jnp.abs(reflected / incident)**2)[::8] * 100,
             "o", ms=4, label="circuit")
axes[1].set(ylabel="Power reflectance (%)")
axes[2].plot(wavelengths_nm, np.unwrap(np.angle(reflection)), label="profile")
axes[2].plot(wavelengths_nm[::8], np.unwrap(np.angle(reflected / incident))[::8],
             "o", ms=4, label="circuit")
axes[2].set(ylabel="Unwrapped reflection phase (rad)")
for ax in axes:
    ax.set(xlabel="Optical carrier wavelength (nm)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Zero-reflection limit and wavelength gradient of the optical conversion.
matched_current, matched_photo = optical_conversion(jnp.sqrt(spectral_power), reflectivity=0.0)
np.testing.assert_allclose(matched_current, jnp.sqrt(spectral_power))
np.testing.assert_allclose(matched_photo, RESPONSIVITY * spectral_power)
spectral_gradient = jax.grad(lambda wl: optical_conversion(
    0.03 + 0.02j, wavelength_nm=wl)[1])(1500.0)
assert bool(jnp.isfinite(spectral_gradient))
assert abs(float(spectral_gradient)) > 1e-12
print("Spectral responsivity, complex reflection, energy balance, and wavelength-gradient checks passed.")


## Bias-dependent S-parameters and AC bandwidth

Use the same device profiles and a small CW optical power (1 µW), holding `wavelength_nm=1550` nm fixed while sweeping **modulation frequency in Hz**. The Gaussian sources are centered at time zero so their DC stamp supplies the required CW operating point for linearization.

Reverse bias reduces the depletion capacitance. With the 50 Ω load, the predicted electrical pole is

$$
f_\mathrm{RC}=\frac{1/R_L+g_\mathrm{dark}}{2\pi C(V_\mathrm{DC})},\qquad
g_\mathrm{dark}=dI_\mathrm{dark}/dV.
$$

First use `circuit.sp()` to sweep the diode electrical port. Convert S11 back to input impedance with $Z_\mathrm{in}=Z_0(1+S_{11})/(1-S_{11})$. Normalizing this impedance yields the response to a small injected current and isolates the **C–V-dependent RC bandwidth**. The reference-port termination used by the S-parameter solver is removed by this conversion; the actual 50 Ω circuit load remains.

Then solve the full linearized circuit for a small optical-power modulation to obtain $\delta I_\mathrm{load}/\delta P_\mathrm{inc}$ in A/W. The diode includes electrical RC filtering; the carrier model adds transport and feedback around its operating point. Because intensity and screening are non-holomorphic, this uses the same full real/imaginary block Jacobian as Circulax's non-holomorphic AC solver. An optical conversion gain in A/W is plotted separately from dimensionless electrical S11.

At these illustrative 20 fF parameters, carrier transport limits the optical bandwidth well before the electrical RC pole. Increasing `c0` in `ac_device()` makes the RC limitation more prominent.

In [ ]:
from circulax.solvers.assembly import assemble_gc_complex_2n
from circulax.solvers.linear import GROUND_STIFFNESS, _build_index_arrays
from circulax.utils import apply_global_params

ac_freqs = jnp.concatenate([jnp.array([0.0]), jnp.logspace(7, 12, 241)])
reverse_biases = [0.5, 1.0, 3.0, 6.0]
ac_power = 1e-6


def ac_device(bias, c0=20e-15, power=ac_power, screening=1.0, tau_rec=500e-12, tau_e0=8e-12, tau_h0=40e-12, base_circuit=circuit):
    groups = apply_global_params(base_circuit.groups, {
        "V": bias, "power": power, "center": 0.0, "screening": screening, "tau_rec": tau_rec,
        "wavelength_nm": REFERENCE_WAVELENGTH_NM, "c0": c0, "tau_e0": tau_e0, "tau_h0": tau_h0})
    device = base_circuit.with_groups(groups)
    op = device.dc(rtol=1e-10, atol=1e-13)
    return device, op


@jax.jit
def solve_ac_blocks(g_matrix, c_matrix, rhs, frequencies):
    return jax.vmap(lambda f: jnp.linalg.solve(
        g_matrix + 2j * jnp.pi * f * c_matrix, rhs))(frequencies)


def optical_ac_response(device, op, frequencies, power=ac_power):
    # Assemble the full non-holomorphic G + j omega C circuit equations.
    n = device.sys_size
    rows, cols, grounds, _ = _build_index_arrays(device.groups, n, is_complex=False)
    rows, cols = jnp.asarray(rows), jnp.asarray(cols)
    g_blocks, c_blocks = assemble_gc_complex_2n(op, device.groups)
    g = jnp.zeros((2 * n, 2 * n))
    c = jnp.zeros_like(g)
    for block, (row_offset, col_offset) in enumerate([(0, 0), (0, n), (n, 0), (n, n)]):
        g = g.at[rows + row_offset, cols + col_offset].add(g_blocks[block])
        c = c.at[rows + row_offset, cols + col_offset].add(c_blocks[block])
    grounded = jnp.concatenate([jnp.asarray(grounds), jnp.asarray(grounds) + n])
    g = g.at[grounded, grounded].add(GROUND_STIFFNESS)

    # Each matched source stamps I = E - 2 sqrt(P). Thus -dF/dP = 1/sqrt(P)
    # at its positive node and the opposite sign at its ground return.
    rhs = jnp.zeros(2 * n)
    ground = device.port_map["GND,p1"]
    for name in ("ideal", "diode", "carrier"):
        optical_node = device.port_map[f"LIGHT_{name},p1"]
        rhs = rhs.at[optical_node].add(1 / jnp.sqrt(power))
        rhs = rhs.at[ground].add(-1 / jnp.sqrt(power))
    small_signal = solve_ac_blocks(g, c, rhs, frequencies)
    # Electrical voltages use the real-envelope block; each value is now a
    # complex modulation phasor. Subtract the actual load return voltage.
    return {name: (small_signal[:, device.port_map[name]] - small_signal[:, ground]) / 50
            for name in ("ideal", "diode", "carrier")}


def bandwidth_3db(frequencies, normalized_response):
    f = np.asarray(frequencies)[1:]
    db = 20 * np.log10(np.abs(normalized_response)[1:])
    target = -10 * np.log10(2)
    crossings = np.flatnonzero((db[:-1] > target) & (db[1:] <= target))
    if not len(crossings):
        return np.nan
    k = crossings[0]
    log_f = np.log10(f[k]) + (target - db[k]) * (np.log10(f[k + 1]) - np.log10(f[k])) / (db[k + 1] - db[k])
    return 10**log_f


fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
print("Reverse bias  C at op (fF)  RC predicted (GHz)  RC AC (GHz)  Optical carrier (GHz)")
ac_results = {}
for k, bias in enumerate(reverse_biases):
    device, op = ac_device(bias)
    v_op = float(jnp.real(device.port(op, "diode") - device.port(op, "BIAS,p1")))
    c_op = float(jax.grad(junction_charge)(v_op))
    g_op = 1 / 50 + float(jax.grad(dark_iv)(v_op))
    predicted_rc = g_op / (2 * np.pi * c_op)
    s11 = np.asarray(device.sp(ports="diode", freqs=ac_freqs, z0=50.0,
                              y_dc=op, holomorphic=False))[:, 0, 0]
    impedance = 50 * (1 + s11) / (1 - s11)
    rc_response = impedance / impedance[0]
    analytic_rc = g_op / (g_op + 2j * np.pi * np.asarray(ac_freqs) * c_op)
    np.testing.assert_allclose(rc_response, analytic_rc, rtol=2e-5, atol=2e-7)
    measured_rc = bandwidth_3db(ac_freqs, rc_response)
    np.testing.assert_allclose(measured_rc, predicted_rc, rtol=0.01)

    optical_response = optical_ac_response(device, op, ac_freqs)
    diode_normalized = np.asarray(optical_response["diode"] / optical_response["diode"][0])
    np.testing.assert_allclose(diode_normalized, analytic_rc, rtol=2e-5, atol=2e-7)
    carrier_normalized = np.asarray(optical_response["carrier"] / optical_response["carrier"][0])
    carrier_bw = bandwidth_3db(ac_freqs, carrier_normalized)
    assert np.isfinite(carrier_bw) and carrier_bw < measured_rc
    np.testing.assert_allclose(optical_response["ideal"], EXTERNAL_RESPONSIVITY, rtol=1e-5)
    ac_results[bias] = {"capacitance": c_op, "rc_bandwidth": measured_rc,
                        "carrier_bandwidth": carrier_bw, "optical_response": optical_response}
    print(f"{bias:12.1f} {c_op * 1e15:13.2f} {predicted_rc / 1e9:19.2f} {measured_rc / 1e9:12.2f} {carrier_bw / 1e9:22.2f}")
    ghz = np.asarray(ac_freqs)[1:] / 1e9
    color = f"C{k}"
    axes[0].semilogx(ghz, 20 * np.log10(np.maximum(np.abs(s11[1:]), 1e-14)), color=color, label=f"{bias:g} V")
    axes[1].semilogx(ghz, 20 * np.log10(np.abs(rc_response[1:])), color=color, label=f"{bias:g} V, {c_op * 1e15:.1f} fF")
    axes[2].semilogx(ghz, 20 * np.log10(np.abs(diode_normalized[1:])), color=color, label=f"{bias:g} V diode")
    axes[2].semilogx(ghz, 20 * np.log10(np.abs(carrier_normalized[1:])), "--", color=color, label=f"{bias:g} V carriers")
axes[0].set(ylabel="Electrical |S11| (dB)", title="Diode electrical reflection")
axes[1].set(ylabel="Normalized transimpedance (dB)", title="C–V-dependent RC bandwidth", ylim=(-25, 1))
axes[2].set(ylabel="Normalized optical conversion gain (dB)", title="RC and carrier transport", ylim=(-40, 1))
for ax in axes:
    ax.set(xlabel="Modulation frequency (GHz)")
    ax.legend(fontsize=7, loc="lower left")
for ax in axes[1:]:
    ax.axhline(-10 * np.log10(2), color="0.5", linestyle=":", linewidth=1)
plt.tight_layout()
plt.show()

assert np.all(np.diff([ac_results[b]["capacitance"] for b in reverse_biases]) < 0)
assert np.all(np.diff([ac_results[b]["rc_bandwidth"] for b in reverse_biases]) > 0)
assert np.all(np.diff([ac_results[b]["carrier_bandwidth"] for b in reverse_biases]) > 0)
print("Bias-dependent S11, RC pole, and optical AC bandwidth checks passed.")


## Screening-dependent AC response at nominal −2 V

Hold the reverse-bias **supply** at 2 V and sweep CW incident power from 1 µW to 0.5 mW at 1550 nm. Solve a new illuminated operating point for each power and apply infinitesimal fractional power modulation. These curves are local AC responses around nonlinear equilibria, not Fourier transforms of large pulses. The table reports the actual junction voltage, including the DC voltage across the 50 Ω load.

Set `tau_rec=1e6` s in **this experiment only** to make recombination negligible and isolate collection-field screening. The range stays below the screening model's steady-state collection limit: without recombination, larger generation above that limit would cause continued carrier accumulation instead of a valid equilibrium. The earlier pulse and saturation examples retain their finite recombination lifetime.

This power sweep uses an explicitly separate **fast-detector scenario**, with reference sweep-out times `tau_e0=1.2` ps and `tau_h0=6` ps, and screening strength `1.5`. These are illustrative settings chosen for a low-power bandwidth around 30 GHz and roughly 10 GHz reduction from 1 µW to 0.5 mW. They are not a fit to measured device data. The earlier pulse examples retain their slower sweep-out times to make electron/hole tails easy to inspect. At −2 V, the bias-dependent sweep-out times are longer than the reference times, which are specified at −3 V.

The left panel retains the original 20 fF junction profile and 50 Ω load. As power tends to zero, the carrier model approaches its **unscreened transport** response; fixed sweep-out times still filter the signal. Increasing power lowers the internal field, lengthens sweep-out times, and reduces the bandwidth.

The right panel adds **10 pF shunt load capacitance**, leaving junction properties, active volume, sweep-out times, and load resistance unchanged. Its RC pole is slower than low-power collection. Here the low-power carrier curve converges to the diode curve through the RC passband, and this fixture remains primarily RC-limited across the power range. The transport-dependent reduction is most visible in the left panel; a sufficiently slow load can mask it. The capacitance is an explicit measurement/load fixture, not a hidden change in absorber geometry.

Both panels use the same gain reference, the low-power external responsivity (0.768 A/W). Full collection conserves the zero-frequency gain even when screening changes the frequency response. The gray control disables screening at 0.5 mW, holding every other parameter fixed. The black dashed curve is the diode at the lowest input power.

The table links internal-field suppression and separate electron/hole sweep-out times to −3 dB bandwidth. Screening lowers the bandwidth in this isolated transport experiment. With strong recombination, a −3 dB width normalized to the model's own DC gain need not decrease monotonically: gain suppression and changing relaxation modes can produce an apparently wider, much weaker response. The earlier finite-recombination examples should therefore be read together with their absolute gain and collected-charge results.

In [ ]:
from copy import deepcopy
from circulax.components.electronic import Capacitor

# A separate, explicitly loaded fixture makes the low-power diode limit visible.
loaded_netlist = deepcopy(netlist)
for name in ("ideal", "diode", "carrier"):
    loaded_netlist["instances"][f"C_load_{name}"] = {
        "component": "load_capacitor", "settings": {"C": 10e-12}}
    loaded_netlist["connections"][f"PD_{name},anode"] = (f"R_{name},p1", f"C_load_{name},p1")
    loaded_netlist["connections"]["GND,p1"] += (f"C_load_{name},p2",)
loaded_circuit = compile_circuit(
    loaded_netlist, {**models, "load_capacitor": Capacitor},
    is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)

power_levels = np.array([1e-6, 1e-5, 1e-4, 3e-4, 5e-4])
transport_tau_rec = 1e6  # negligible recombination, matching the screening-only pulse experiment
fixed_reverse_supply = 2.0
# A separate fast-detector scenario: approximately 30 GHz at low power,
# with an illustrative 10 GHz screening-induced drop across this power range.
transport_settings = {"tau_e0": 1.2e-12, "tau_h0": 6e-12, "screening": 1.5}
power_ac_results = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
print("Load C (pF)  P (mW)  Junction (V)  Field / E0  tau_e (ps)  tau_h (ps)  Gain (A/W)  BW (GHz)")

for row, (label, fixture, load_cap_pf) in enumerate([
    ("Fast-detector scenario: 50 Ω load", circuit, 0.0),
    ("50 Ω load + 10 pF fixture capacitance", loaded_circuit, 10.0),
]):
    results = {}
    for k, power in enumerate(power_levels):
        device, op = ac_device(fixed_reverse_supply, power=power, tau_rec=transport_tau_rec, base_circuit=fixture, **transport_settings)
        response = optical_ac_response(device, op, ac_freqs, power=power)
        carrier = np.asarray(response["carrier"])
        normalized = carrier / carrier[0]
        v = device.port(op, "carrier") - device.port(op, "BIAS,p1")
        x_e = device.port(op, "PD_carrier,x_e")
        x_h = device.port(op, "PD_carrier,x_h")
        e0, field = internal_field(x_e, x_h, v, screening=transport_settings["screening"])
        tau_e, tau_h, _ = carrier_rates(x_e, x_h, v,
            tau_e0=transport_settings["tau_e0"], tau_h0=transport_settings["tau_h0"],
            screening=transport_settings["screening"], tau_rec=transport_tau_rec)
        gain = float(np.real(carrier[0]))
        bw = bandwidth_3db(ac_freqs, normalized)
        assert gain > 0 and np.isfinite(bw)
        results[power] = {"response": response, "gain": gain, "bandwidth": bw,
                          "field_ratio": float(field / e0), "junction_v": float(jnp.real(v))}
        print(f"{load_cap_pf:11.0f} {power * 1e3:8.3g} {float(jnp.real(v)):13.4f} "
              f"{float(field / e0):11.3f} {float(tau_e) * 1e12:11.2f} {float(tau_h) * 1e12:11.2f} "
              f"{gain:11.4f} {bw / 1e9:9.3f}")
        ghz = np.asarray(ac_freqs)[1:] / 1e9
        axes[row].semilogx(ghz, 20 * np.log10(np.abs(carrier[1:]) / EXTERNAL_RESPONSIVITY),
                           color=f"C{k}", label=f"{power * 1e3:g} mW")

    # Compare the identical high-power circuit with only screening disabled.
    high_power = power_levels[-1]
    no_screen_device, no_screen_op = ac_device(
        fixed_reverse_supply, power=high_power, tau_rec=transport_tau_rec, base_circuit=fixture,
        **{**transport_settings, "screening": 0.0})
    no_screen_response = optical_ac_response(no_screen_device, no_screen_op, ac_freqs, power=high_power)
    no_screen_carrier = np.asarray(no_screen_response["carrier"])
    diode = np.asarray(results[power_levels[0]]["response"]["diode"])
    axes[row].semilogx(ghz, 20 * np.log10(np.abs(diode[1:]) / EXTERNAL_RESPONSIVITY), "--",
                       color="black", lw=2, label="low-power diode reference")
    axes[row].semilogx(ghz, 20 * np.log10(np.abs(no_screen_carrier[1:]) / EXTERNAL_RESPONSIVITY), ":",
                       color="0.4", lw=2, label="0.5 mW, screening off")
    axes[row].set(title=label, ylim=(-40, 1), xlim=(0.1, 500) if row == 0 else (0.01, 20),
                  ylabel="Differential gain / external responsivity (dB)", xlabel="Modulation frequency (GHz)")
    axes[row].legend(fontsize=8, loc="lower left")
    axes[row].axhline(-10 * np.log10(2), color="0.5", ls=":", lw=1)

    assert results[high_power]["field_ratio"] < results[power_levels[0]]["field_ratio"]
    np.testing.assert_allclose([results[p]["gain"] for p in power_levels], EXTERNAL_RESPONSIVITY, rtol=1e-4)
    assert np.all(np.diff([results[p]["bandwidth"] for p in power_levels]) < 0)
    assert results[high_power]["bandwidth"] < bandwidth_3db(
        ac_freqs, no_screen_carrier / no_screen_carrier[0])
    power_ac_results[load_cap_pf] = results

plt.tight_layout()
plt.show()

# Quantify the low-power convergence in the intentionally RC-limited fixture.
loaded_results = power_ac_results[10.0]
low_response = loaded_results[power_levels[0]]["response"]
low_carrier = np.asarray(low_response["carrier"] / low_response["carrier"][0])
low_diode = np.asarray(low_response["diode"] / low_response["diode"][0])
diode_bandwidth = bandwidth_3db(ac_freqs, low_diode)
passband = np.asarray(ac_freqs) <= diode_bandwidth
low_power_difference_db = np.max(np.abs(20 * np.log10(np.abs(low_carrier[passband] / low_diode[passband]))))
assert low_power_difference_db < 0.1
np.testing.assert_allclose(loaded_results[power_levels[0]]["bandwidth"], diode_bandwidth, rtol=0.02)
assert loaded_results[power_levels[-1]]["bandwidth"] < loaded_results[power_levels[0]]["bandwidth"]
print(f"RC-limited low-power carrier/diode difference through the diode passband: {low_power_difference_db:.3f} dB")
print(f"RC-limited diode bandwidth: {diode_bandwidth / 1e9:.3f} GHz")
print("Fixed-bias power sweep, low-power diode limit, and screening-control checks passed.")

low_bw = power_ac_results[0.0][power_levels[0]]["bandwidth"]
high_bw = power_ac_results[0.0][power_levels[-1]]["bandwidth"]
bandwidth_drop_ghz = (low_bw - high_bw) / 1e9
print(f"Illustrative fast-detector bandwidth: {low_bw / 1e9:.2f} → {high_bw / 1e9:.2f} GHz")
print(f"Screening-induced reduction across the power sweep: {bandwidth_drop_ghz:.2f} GHz")
assert 8 < bandwidth_drop_ghz < 12


## With and without pad/metal parasitics

Build a reusable **photodetector subcircuit**, then instantiate it in the full optical-source/bias/load circuit. The package owns the intrinsic detector, a true series resistor, and a pad parallel R–C network. The illustrative values are configurable: **20 Ω contact/metal series resistance**, **100 fF pad capacitance**, and **1 MΩ pad shunt resistance**. The pad is a parallel R–C network connected from the external output to ground; the existing 50 Ω measurement load is separate.

```text
intrinsic PD anode ── R_series ── output ── R_load (50 Ω) ── ground
                                  ├────── R_pad ─────────── ground
                                  └────── C_pad ─────────── ground
PD cathode ── +2 V bias supply
```

`make_photodetector_subcircuit()` returns a compiled Circulax `Circuit` with physical ports `optical_in`, `anode`, `cathode`, and `pad_ground`. The independent pad ground keeps pad capacitance referenced to ground rather than to the reverse-biased cathode. `make_parasitic_fixture()` puts these compiled packages in its models mapping and instantiates them as components in the full testbench; Circulax expands their hierarchy during compilation. It leaves the original device-only circuit unchanged. The I–V, C–V, and carrier equations automatically see the **intrinsic** anode-to-cathode voltage, before the series resistor. The junction anode is a distinct internal node (`PD_carrier~CORE,anode` after hierarchy expansion), connected to the external anode only through `R_series`. The topology checks verify that the resistor has two distinct nodes and is not bypassed. Named output ports map to the external load node; AC gain measures current delivered to the 50 Ω load, excluding current diverted through the pad network.

Repeat the same power sweep and fast-detector parameters as above. The frequency plot compares the lowest and highest powers, with dashed curves for the device-only circuit and solid curves for the circuit with parasitics. The bandwidth plot includes every power level and the diode reference with the same parasitics. A screening-disabled control separates transport feedback from electrical filtering. Turning off only `C_pad` verifies that it affects AC response without changing the DC operating point.

The DC gain division due to pad leakage is $R_\mathrm{pad}/(R_\mathrm{pad}+50\,\Omega)$. The series drop and pad capacitance produce additional frequency dependence; the full circuit solve accounts for their interaction with junction capacitance and carrier storage. These values and the targeted 10 GHz **device-only** bandwidth reduction remain illustrative rather than device calibration.

Setting `with_parasitics=False` gives a package containing only the intrinsic core, with its original three physical ports. It removes the resistor and pad branches entirely. Checks at both ends of the power sweep verify that embedding this device-only package produces the same AC response as the original intrinsic-component circuit.

In [ ]:
def make_photodetector_subcircuit(model_kind="carrier", series_resistance=20.0,
                                  pad_resistance=1e6, pad_capacitance=100e-15, with_parasitics=True):
    """Reusable PD package: intrinsic core, true series R, and pad R || C.

    Physical ports: optical_in, anode (external), cathode, and pad_ground.
    The core's junction anode is an internal node on the other side of R_series.
    """
    if model_kind not in ("ideal", "diode", "carrier"):
        raise ValueError("model_kind must be ideal, diode, or carrier")
    if not with_parasitics:
        return compile_circuit({
            "instances": {"CORE": {"component": model_kind}}, "connections": {},
            "ports": {port: f"CORE,{port}" for port in PORTS},
        }, models, is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)
    package_netlist = {
        "instances": {
            "CORE": {"component": model_kind},
            "R_series": {"component": "load", "settings": {"R": series_resistance}},
            "R_pad": {"component": "load", "settings": {"R": pad_resistance}},
        },
        "connections": {},
        "nets": [
            {"p1": "CORE,anode", "p2": "R_series,p1"},
            {"p1": "R_series,p2", "p2": "R_pad,p1"},
        ],
        "ports": {
            "optical_in": "CORE,optical_in",
            "anode": "R_series,p2",
            "cathode": "CORE,cathode",
            "pad_ground": "R_pad,p2",
        },
    }
    if pad_capacitance > 0:
        package_netlist["instances"]["C_pad"] = {
            "component": "pad_capacitor", "settings": {"C": pad_capacitance}}
        package_netlist["nets"].extend([
            {"p1": "R_series,p2", "p2": "C_pad,p1"},
            {"p1": "R_pad,p2", "p2": "C_pad,p2"},
        ])
    return compile_circuit(package_netlist, {**models, "pad_capacitor": Capacitor},
                           is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)


def make_parasitic_fixture(series_resistance=20.0, pad_resistance=1e6, pad_capacitance=100e-15, with_parasitics=True):
    # The full circuit contains PD package instances plus sources and loads.
    # Circulax accepts compiled Circuit objects directly in the models mapping.
    fixture_netlist = deepcopy(netlist)
    fixture_models = dict(models)
    for name in ("ideal", "diode", "carrier"):
        package_name = f"packaged_{name}"
        fixture_models[package_name] = make_photodetector_subcircuit(
            name, series_resistance, pad_resistance, pad_capacitance, with_parasitics=with_parasitics)
        fixture_netlist["instances"][f"PD_{name}"] = {"component": package_name}
        if with_parasitics:
            fixture_netlist["connections"]["GND,p1"] += (f"PD_{name},pad_ground",)
        # Existing optical, bias, anode-load, and output-port connections now
        # address the package's physical ports, rather than individual parts.
    return compile_circuit(fixture_netlist, fixture_models,
                           is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)


series_resistance = 20.0
pad_resistance = 1e6
pad_capacitance = 100e-15
parasitic_circuit = make_parasitic_fixture(series_resistance, pad_resistance, pad_capacitance)
# Verify topology: R_series connects two distinct nodes, with no bypass.
port_map = parasitic_circuit.port_map
assert port_map["PD_carrier~CORE,anode"] == port_map["PD_carrier~R_series,p1"]
assert port_map["carrier"] == port_map["PD_carrier~R_series,p2"]
assert port_map["carrier"] == port_map["PD_carrier~R_pad,p1"] == port_map["PD_carrier~C_pad,p1"]
assert port_map["PD_carrier~CORE,anode"] != port_map["carrier"]
parasitic_ac_results = {}
print("P (mW)  Device BW (GHz)  With parasitics (GHz)  Diode+parasitics (GHz)  Junction (V)  Series drop (mV)")
for power in power_levels:
    device, op = ac_device(fixed_reverse_supply, power=power, tau_rec=transport_tau_rec,
                           base_circuit=parasitic_circuit, **transport_settings)
    response = optical_ac_response(device, op, ac_freqs, power=power)
    carrier = np.asarray(response["carrier"])
    diode = np.asarray(response["diode"])
    carrier_bw = bandwidth_3db(ac_freqs, carrier / carrier[0])
    diode_bw = bandwidth_3db(ac_freqs, diode / diode[0])
    junction_v = device.port(op, "PD_carrier~CORE,anode") - device.port(op, "BIAS,p1")
    external_v = device.port(op, "carrier") - device.port(op, "GND,p1")
    series_drop = device.port(op, "PD_carrier~CORE,anode") - device.port(op, "carrier")
    expected_series_drop = series_resistance * external_v * (1 / 50 + 1 / pad_resistance)
    np.testing.assert_allclose(series_drop, expected_series_drop, rtol=1e-7, atol=1e-12)
    expected_load_gain = EXTERNAL_RESPONSIVITY * pad_resistance / (pad_resistance + 50)
    np.testing.assert_allclose(carrier[0], expected_load_gain, rtol=1e-4)
    bare_bw = power_ac_results[0.0][power]["bandwidth"]
    assert 0 < carrier_bw < bare_bw
    parasitic_ac_results[power] = {"response": response, "bandwidth": carrier_bw,
                                  "diode_bandwidth": diode_bw, "op": op, "device": device}
    print(f"{power * 1e3:7.3g} {bare_bw / 1e9:16.2f} {carrier_bw / 1e9:22.2f} "
          f"{diode_bw / 1e9:23.2f} {float(jnp.real(junction_v)):13.4f} {float(jnp.real(series_drop)) * 1e3:17.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ghz = np.asarray(ac_freqs)[1:] / 1e9
for color, power in [("C0", power_levels[0]), ("C4", power_levels[-1])]:
    for label, response, linestyle in [
        ("without", power_ac_results[0.0][power]["response"]["carrier"], "--"),
        ("with", parasitic_ac_results[power]["response"]["carrier"], "-"),
    ]:
        axes[0].semilogx(ghz, 20 * np.log10(np.abs(response[1:]) / EXTERNAL_RESPONSIVITY),
                         linestyle=linestyle, color=color, lw=2,
                         label=f"{power * 1e3:g} mW, {label} parasitics")

no_screen_device, no_screen_op = ac_device(
    fixed_reverse_supply, power=power_levels[-1], tau_rec=transport_tau_rec,
    base_circuit=parasitic_circuit, **{**transport_settings, "screening": 0.0})
no_screen = optical_ac_response(no_screen_device, no_screen_op, ac_freqs, power=power_levels[-1])["carrier"]
assert bandwidth_3db(ac_freqs, no_screen / no_screen[0]) > parasitic_ac_results[power_levels[-1]]["bandwidth"]
axes[0].semilogx(ghz, 20 * np.log10(np.abs(no_screen[1:]) / EXTERNAL_RESPONSIVITY), ":",
                 color="0.4", label="0.5 mW, parasitics on, screening off")
axes[0].set(xlabel="Modulation frequency (GHz)", ylabel="Gain / external responsivity (dB)",
            title="Transport and parasitics at nominal −2 V", xlim=(0.1, 200), ylim=(-35, 1))
axes[0].axhline(-10 * np.log10(2), color="0.5", ls=":", lw=1)
axes[0].legend(fontsize=7, loc="lower left")

powers_mw = power_levels * 1e3
axes[1].semilogx(powers_mw, [power_ac_results[0.0][p]["bandwidth"] / 1e9 for p in power_levels],
                 "o--", label="without parasitics")
axes[1].semilogx(powers_mw, [parasitic_ac_results[p]["bandwidth"] / 1e9 for p in power_levels],
                 "o-", label="with parasitics")
axes[1].semilogx(powers_mw, [parasitic_ac_results[p]["diode_bandwidth"] / 1e9 for p in power_levels],
                 ":", color="0.3", label="diode with parasitics")
axes[1].set(xlabel="CW incident power (mW)", ylabel="−3 dB bandwidth (GHz)",
            title="Power sweep with the same intrinsic device")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# C_pad is reactive: removing just that branch leaves the DC solution intact.
no_cap_circuit = make_parasitic_fixture(series_resistance, pad_resistance, pad_capacitance=0.0)
no_cap_device, no_cap_op = ac_device(
    fixed_reverse_supply, power=power_levels[0], tau_rec=transport_tau_rec,
    base_circuit=no_cap_circuit, **transport_settings)
no_cap_response = optical_ac_response(no_cap_device, no_cap_op, ac_freqs, power=power_levels[0])["carrier"]
with_cap = parasitic_ac_results[power_levels[0]]
np.testing.assert_allclose(no_cap_device.port(no_cap_op, "carrier"),
                           with_cap["device"].port(with_cap["op"], "carrier"), rtol=1e-7, atol=1e-12)
np.testing.assert_allclose(no_cap_response[0], with_cap["response"]["carrier"][0], rtol=1e-7)
assert bandwidth_3db(ac_freqs, no_cap_response / no_cap_response[0]) > with_cap["bandwidth"]
print("Subcircuit topology, series voltage drop, pad leakage division, capacitor DC invariance, and parasitic bandwidth checks passed.")


# The same package interface also supports a true device-only topology:
# with_parasitics=False removes R_series, R_pad, and C_pad entirely.
bare_package_circuit = make_parasitic_fixture(with_parasitics=False)
assert "PD_carrier~R_series,p1" not in bare_package_circuit.port_map
for power in (power_levels[0], power_levels[-1]):
    bare_device, bare_op = ac_device(
        fixed_reverse_supply, power=power, tau_rec=transport_tau_rec,
        base_circuit=bare_package_circuit, **transport_settings)
    bare_response = optical_ac_response(bare_device, bare_op, ac_freqs, power=power)
    for name in ("ideal", "diode", "carrier"):
        np.testing.assert_allclose(bare_response[name], power_ac_results[0.0][power]["response"][name],
                                   rtol=1e-6, atol=1e-9)
print("Packaged device-only response matches the intrinsic component circuit.")


## Next steps

Fit complex optical reflection and external responsivity versus wavelength, separating absorption loss from conversion efficiency. Then fit dark I–V and C–V, then low-power carrier sweep-out times, then screening and recombination using power and bias sweeps. Measured profiles, series resistance, trapping, avalanche multiplication, and spatial drift–diffusion are extensions beyond this compact example. The present two-population approximation does not reproduce a spatial transit-time distribution; its short-pulse rise is partly set by the stimulus and load RC. Measured small-signal bandwidth and repeated-pulse recovery are useful additional calibration targets.

Once the profiles and transport approximation are agreed, promote the intrinsic decorated components, shared helpers, and reusable photodetector-package factory into the library, with this notebook as the worked example. Keep the intrinsic junction model separate from the package circuit so contacts, metal, pads, bias networks, and loads can be changed independently.